# PFE ML — Phase C: Temporal Stability Of The Tuned HGB Winner

Phase B identified HistGradientBoosting (HGB) as the operational winner. Its 2024-test metrics were AP ≈ 0.155, AUC ≈ 0.80, F1 @ 0.5 ≈ 0.15. Phase B's headline number rests on a *single* held-out year — 2024.

**Phase C question:** is that single-year result a one-off, or does HGB perform comparably when the held-out year is different?

If the metrics swing wildly when we shift the test year, the 2024 number reflects a year-specific quirk and is not a reliable estimate of forward performance. If the metrics stay close to AP 0.155 / AUC 0.80 across multiple test years, the model is temporally stable.

## Methodology

- **Same model, same features, same hyperparameters** as the Phase B HGB tuned winner — only the time-split changes.
- **Walk-forward backtest:** for each test year in {2022, 2023, 2024}, retrain the HGB pipeline on all years strictly before it, then evaluate on the held-out year. This is implemented by passing `--train-end-year <Y>` to `train_continuity_model.py`, which truncates the data so year `Y` is the latest (and therefore the test year).
- **Sample:** same 2M-row deterministic-hash sample used in every prior run.
- **Metrics:** AP, AUC, F1@0.5 reported per test year; we look for tight clustering rather than monotone improvement.

## Expected outcome

- **Stable model** → AP and AUC stay within ±1 pp of the 2024 number for both alternate test years.
- **Year-specific quirk** → one or both alternate years show ≥2 pp degradation in AP. This would call for inspecting label rate per year, feature drift, and event rarity in that period.

## What this notebook produces

- Three new `runs/*/` artifact folders (one per test year) appended to `model_run_comparison.csv`.
- `temporal_phase_c/temporal_stability.csv` — three-row table comparing AP / AUC / F1 across test years.
- `temporal_phase_c/temporal_stability.png` — per-test-year metric chart with the Phase B baseline overlaid.

## 1. Runtime And Constants

CPU runtime is sufficient — HGB does not use GPU. Each retrain is a single fit on ~1.4–1.7M rows of training data and should complete in roughly 6–10 minutes. Total notebook runtime ≈ 25–35 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
ARTIFACTS_DIR = f'{DRIVE_ROOT}/ml-artifacts'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

TARGET = 'continuity_risk_12m_label'
TRAIN_MAX_ROWS = 2_000_000
TRAIN_START_YEAR = 2017

# Test years to evaluate. Each one is used as the *latest* year in the data
# (so train_continuity_model.py holds it out as the test split). Years are
# picked to give enough training history before each one.
TEST_YEARS = [2022, 2023, 2024]

# Phase B winner. The notebook will refuse to run without this file present.
TUNED_HGB_PARAMS = Path(ARTIFACTS_DIR) / 'tuned_params_hgb.json'

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('BACKEND_DIR  =', BACKEND_DIR)
print('DRIVE_ROOT   =', DRIVE_ROOT)
print('TEST_YEARS   =', TEST_YEARS)
print('HGB params   =', TUNED_HGB_PARAMS)

## 2. Pull Code And Install Dependencies

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

In [ ]:
import json

if not TUNED_HGB_PARAMS.exists():
    raise SystemExit(
        f'Phase B output missing: {TUNED_HGB_PARAMS}.\n'
        f'Run Phase B first (collabs/pfe_ml_colab_tuning_phase_b.ipynb).'
    )

tuned_params = json.loads(TUNED_HGB_PARAMS.read_text(encoding='utf-8'))
print('Phase B HGB tuned params:')
for k, v in tuned_params.items():
    print(f'  {k}: {v}')

## 3. Run The Walk-Forward Backtest

Each iteration invokes `train_continuity_model.py` with `--train-end-year <Y>`. This truncates the data so year `Y` is the latest available, which the training script then automatically uses as the held-out test year (train on all years strictly before it).

The `--params-file` flag injects the Phase B tuned hyperparameters. The same 2M-row deterministic-hash sample is used for every run, so the only varying input is the time-split point.

In [ ]:
import shlex, subprocess, sys, time

phase_c_runs = {}
for test_year in TEST_YEARS:
    print('=' * 72)
    print(f'Phase C — HGB tuned, test year = {test_year}')
    print('=' * 72)
    cmd = [
        sys.executable, '-u',
        '-m', 'app.tools.train_continuity_model',
        '--data-lake-dir', DATA_LAKE,
        '--artifacts-dir', ARTIFACTS_DIR,
        '--target', TARGET,
        '--train-start-year', str(TRAIN_START_YEAR),
        '--train-end-year', str(test_year),
        '--max-rows', str(TRAIN_MAX_ROWS),
        '--min-rows', '1000',
        '--model-family', 'hgb',
        '--params-file', str(TUNED_HGB_PARAMS),
    ]
    print(' '.join(shlex.quote(p) for p in cmd))
    start = time.time()
    subprocess.run(cmd, check=True)
    elapsed = time.time() - start
    metadata = json.loads((Path(ARTIFACTS_DIR) / 'model_metadata.json').read_text(encoding='utf-8'))
    phase_c_runs[test_year] = {
        'run_name': metadata['run_name'],
        'run_dir': metadata['run_artifacts_dir'],
        'elapsed_seconds': elapsed,
        'metrics': metadata['metrics'],
        'train_rows': metadata.get('rows'),
    }
    m = metadata['metrics']
    print(
        f"\nDone ({elapsed:.0f}s). test={test_year}  "
        f"AUC={m['roc_auc']:.4f}  AP={m['average_precision']:.4f}  "
        f"F1@0.5={m['f1_at_0_5']:.4f}\n"
    )

print('All Phase C retrains complete.')

## 4. Temporal Stability Table

Pull the per-year metrics into a single dataframe and compute spread (max − min) across test years. A spread under 1 pp on AP and AUC is strong evidence the model is not year-specific.

In [ ]:
import pandas as pd

rows = []
for test_year, info in sorted(phase_c_runs.items()):
    m = info['metrics']
    rows.append({
        'test_year': test_year,
        'run_name': info['run_name'],
        'train_rows': info['train_rows'],
        'average_precision': m['average_precision'],
        'roc_auc': m['roc_auc'],
        'precision_at_0_5': m['precision_at_0_5'],
        'recall_at_0_5': m['recall_at_0_5'],
        'f1_at_0_5': m['f1_at_0_5'],
    })
stability_df = pd.DataFrame(rows)

print('Temporal stability — HGB tuned, walk-forward backtest:')
print(stability_df.to_string(index=False))

spread = {
    'AP':  stability_df['average_precision'].max() - stability_df['average_precision'].min(),
    'AUC': stability_df['roc_auc'].max() - stability_df['roc_auc'].min(),
    'F1':  stability_df['f1_at_0_5'].max() - stability_df['f1_at_0_5'].min(),
}
print('\nSpread across test years (max − min):')
for k, v in spread.items():
    print(f'  {k}: {v:+.4f}')

phase_c_dir = Path(ARTIFACTS_DIR) / 'temporal_phase_c'
phase_c_dir.mkdir(parents=True, exist_ok=True)
stability_csv = phase_c_dir / 'temporal_stability.csv'
stability_df.to_csv(stability_csv, index=False)
print(f'\nSaved: {stability_csv}')

## 5. Plot Per-Test-Year Metrics

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = [('average_precision', 'Average Precision'),
                   ('roc_auc', 'ROC AUC'),
                   ('f1_at_0_5', 'F1 @ 0.5')]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (metric, label) in zip(axes, metrics_to_plot):
    ax.plot(stability_df['test_year'], stability_df[metric],
            marker='o', color='#0f766e', linewidth=2)
    for x, y in zip(stability_df['test_year'], stability_df[metric]):
        ax.annotate(f'{y:.4f}', (x, y), textcoords='offset points', xytext=(0, 8),
                    ha='center', fontsize=9)
    ymin = stability_df[metric].min()
    ymax = stability_df[metric].max()
    pad = max(0.005, (ymax - ymin) * 0.5)
    ax.set_ylim(ymin - pad, ymax + pad * 2)
    ax.set_xticks(stability_df['test_year'])
    ax.set_xlabel('Test year')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.grid(True, alpha=0.3)
fig.suptitle('Phase C — HGB tuned, walk-forward temporal backtest')
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(phase_c_dir / 'temporal_stability.png', dpi=160)
plt.show()

## 6. Decision Criteria And Thesis Framing

### Read the spread table from section 4

- **AP spread < 0.01 (1 pp), AUC spread < 0.01** → temporally stable. Phase B's 2024-test headline is representative of forward performance. Proceed to Phase D (interpretability).
- **AP spread 0.01–0.02 (1–2 pp)** → mild year sensitivity. Acceptable for a thesis result but worth a paragraph attributing it to per-year label rarity / regulatory cohort effects. Compare `test_positive_rate` across years in `model_run_comparison.csv`.
- **AP spread > 0.02** → year-specific behaviour. Phase B's 2024 number is not a clean forward estimate. Investigate before proceeding: per-year label rates, feature distributions, the COVID-era event spike (2020–2021), and whether one test year contains an anomaly.

### Thesis framing for Phase C

*"Temporal stability was assessed by walk-forward backtesting: the tuned HGB model was retrained for each test year in {2022, 2023, 2024}, training only on years strictly prior to the test year and evaluating on the held-out year. The 2M-row hash-deterministic sample and tuned hyperparameters from Phase B were reused unchanged. The spread of average precision across the three test years was {AP_spread:.3f} (AUC spread {AUC_spread:.3f}), confirming that the Phase B 2024 result is representative of forward performance rather than a year-specific artefact."*

(Fill in the actual spread numbers from section 4 when transferring to the thesis.)

### What's next

**Phase D (interpretability):** SHAP values on the tuned-HGB final model. Tells the reader *why* the predictions are what they are, not just that they perform. The features that drove Phase A's performance (`administrative_status_at_cutoff`, `days_since_last_legal_event`, `company_age_years`, `radiation_events_count_all`) are the candidates whose SHAP distributions are most worth examining.